# Chicago Crime · What Citywide Totals Can Hide
**Hesub Chaka | SQL analysis and visualization**

Reported crime is often summarized with a single citywide number. This analysis follows that number into three different views: the share of records associated with arrest, the seasonal distribution within each year, and differences across police districts.

The goal is descriptive: identify patterns worth investigating without treating arrests as convictions, counts as population-adjusted risk, or linear trends as causal effects.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'scripts').is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
import pandas as pd
from analyze import load, SQL_ANNUAL, SQL_TRENDS
frames, db = load()


## 1. Annual volume and arrest share
A SQL window calculates trailing five-year averages. For 2001–2004 the window expands from one to four observations; thereafter it contains five years. Partial 2025 is excluded from the chart and trend comparisons.

In [2]:
print(SQL_ANNUAL)
annual = pd.read_sql_query(SQL_ANNUAL, db)
annual.iloc[[0,-1]].round(2)


SELECT year, incidents, arrests,
 100.0*arrests/NULLIF(incidents,0) AS arrest_pct,
 AVG(incidents) OVER (ORDER BY year ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS incidents_ma5,
 AVG(arrests) OVER (ORDER BY year ROWS BETWEEN 4 PRECEDING AND CURRENT ROW) AS arrests_ma5
FROM annual WHERE year < 2025 ORDER BY year;



,year,incidents,arrests,arrest_pct,incidents_ma5,arrests_ma5
0,2001,485979,141955,29.21,485979.0,141955.0
23,2024,259667,35900,13.83,237425.6,31461.2


In [3]:
from analyze import run
frames, annual, trends, summary = run()

{
  "total_records_in_scope": 8459600,
  "distinct_cases": 8458992,
  "first_year": 2001,
  "last_full_year": 2024,
  "incident_change_pct": -46.56826735311608,
  "first_arrest_pct": 29.210109901868186,
  "last_arrest_pct": 13.825399453915978,
  "city_slope_2010_2024": -8468.335714285735,
  "districts_positive_slope": 1,
  "districts_negative_slope": 21,
  "districts_complete": 22,
  "summer_relative_intensity": 1.0857716396525356,
  "winter_relative_intensity": 0.9032179871566033
}
All reconciliation and regression checks passed.


![Analysis chart](../figures/01_annual_trends.png)

### Finding
The public snapshot contains **46.6% fewer reported records in 2024 than in 2001**. The share with `arrest = true` falls from **29.2% to 13.8%**. This is a change in recorded attributes, not a clearance rate or evidence of a particular cause. More recent incidents also have less time for later updates.

## 2. Is there a seasonal pattern after adjusting for month length?
A 31-day month has more opportunity for records than February. Each cell therefore compares daily records in that month with that year’s daily average. This also prevents high-volume early years from dominating the color scale.

![Analysis chart](../figures/02_seasonality.png)

### Finding
Across the 24 full years, June–August average **1.09 times** their year’s daily average, versus **0.90 times** for December–February. These values average the month/year ratios equally. This pattern does not identify the role of weather or other causes.

## 3. Do high-volume categories also have high arrest shares?
Use a single complete year to compare category volumes and recorded arrest shares with a consistent time window.

In [6]:
categories = pd.read_sql_query("SELECT primary_type, incidents, arrests, ROUND(100.0*arrests/incidents,1) AS arrest_pct FROM category_annual WHERE year=2024 ORDER BY incidents DESC LIMIT 10", db)
categories

,primary_type,incidents,arrests,arrest_pct
0,THEFT,60561,4129,6.8
1,BATTERY,46188,7906,17.1
2,CRIMINAL DAMAGE,28557,1109,3.9
3,ASSAULT,23488,2535,10.8
4,MOTOR VEHICLE THEFT,21713,612,2.8
5,OTHER OFFENSE,17169,3299,19.2
6,DECEPTIVE PRACTICE,16481,540,3.3
7,ROBBERY,9120,765,8.4
8,BURGLARY,8436,468,5.5
9,WEAPONS VIOLATION,7841,4809,61.3


![Analysis chart](../figures/03_categories.png)

### Finding
Theft is the largest recorded category in 2024, while weapons violations have a much higher recorded arrest share among these ten categories. Category differences can reflect reporting and enforcement practices as well as the nature of the offenses. Arrest share does not measure severity or neighborhood safety.

## 4. Do district trends agree with the citywide trend?
The SQL query uses centered sums to calculate an ordinary least-squares slope and R² for 2010–2024. A district must have observations in all 15 years. Codes outside 01–25 and missing codes remain in city totals but are excluded from the district comparison; retired or incomplete codes do not receive fabricated zero years.

For comparison on the chart, the slope is divided by each district’s mean annual count. This is **not** compound annual growth. Historical boundary changes and classification changes still limit interpretation.

In [8]:
print(SQL_TRENDS)
trends.round(3)


WITH moments AS (
 SELECT district, COUNT(*) AS years,
 AVG(year*1.0) AS xbar, AVG(incidents*1.0) AS ybar,
 SUM(year*year*1.0)-SUM(year*1.0)*SUM(year*1.0)/COUNT(*) AS sxx,
 SUM(year*incidents*1.0)-SUM(year*1.0)*SUM(incidents*1.0)/COUNT(*) AS sxy,
 SUM(incidents*incidents*1.0)-SUM(incidents*1.0)*SUM(incidents*1.0)/COUNT(*) AS syy
 FROM district_annual
 WHERE year BETWEEN 2010 AND 2024 AND district BETWEEN 1 AND 25
 GROUP BY district
)
SELECT district, years, ybar AS mean_incidents,
 sxy/NULLIF(sxx,0) AS slope,
 sxy*sxy/NULLIF(sxx*syy,0) AS r_squared,
 100.0*sxy/NULLIF(sxx*ybar,0) AS slope_pct_of_mean
FROM moments WHERE years=15 ORDER BY slope_pct_of_mean;



,district,years,mean_incidents,slope,r_squared,slope_pct_of_mean
0,7.0,15,14972.333,-861.486,0.915,-5.754
1,15.0,15,11305.600,-607.746,0.925,-5.376
2,25.0,15,15322.400,-705.289,0.750,-4.603
3,9.0,15,12912.400,-584.954,0.752,-4.530
4,8.0,15,18160.733,-725.536,0.656,-3.995
5,11.0,15,18270.667,-705.346,0.911,-3.861
6,14.0,15,10050.467,-387.321,0.570,-3.854
7,5.0,15,12211.533,-452.814,0.832,-3.708
8,4.0,15,15940.867,-572.532,0.752,-3.592
9,3.0,15,13924.133,-492.471,0.631,-3.537


![Analysis chart](../figures/04_district_trends.png)

### Finding
The citywide fitted slope is **-8,468 records per year** over 2010–2024. Of the 22 complete district series in this comparison, **21 have negative slopes and 1 has a positive slope**. District 01’s R² is only about 0.03, so its positive straight-line slope explains little of its year-to-year variation. Opposite slope signs alone do not establish Simpson’s paradox.

SQL slopes are checked independently against NumPy fits before publishing the figures.

## Conclusions and limits
Citywide totals describe a broad decline in this recorded series, while arrest shares, category composition, seasonality and district patterns add different information. The analysis supports follow-up questions rather than causal or safety rankings.

- No population or reporting-exposure adjustment: comparisons describe records, not individual risk.
- Arrest flags may change and are not convictions, arrest event counts, or clearance rates.
- Source updates, policing/reporting practices, classification changes, and district boundaries can affect trends.
- 2025 is incomplete and excluded from full-year charts and regressions.
- Aggregates cannot validate every underlying row; the live public source can change after this saved snapshot was retrieved.

Originally developed as an academic SQL/database project. AI assisted query structuring, code revision and presentation; executed outputs and independent checks support the numerical findings in this edition.